# Quantum ESPRESSO 7.6 GPU on Google Colab (NVIDIA T4)
Target: Li3SiNi3O8, task 34911

This notebook:
1. Checks that Colab actually assigned an NVIDIA T4.
2. Installs NVIDIA HPC SDK 25.5 (CUDA 12.9).
3. Builds Quantum ESPRESSO 7.6 with NVIDIA GPU/OpenACC support for compute capability 7.5.
4. Uploads the previously prepared `Li3SiNi3O8_QE_inputs.tar.gz`.
5. Downloads the PBE PAW pseudopotentials.
6. Runs the magnetic PBE+U SCF calculation.
7. Prints convergence and magnetization information.

## Before running
In Colab choose:

**Runtime → Change runtime type → Hardware accelerator → T4 GPU**

Free Colab GPU type/availability is not guaranteed. The first cell must report **Tesla T4**. The T4 has compute capability 7.5.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvidia-smi --query-gpu=compute_cap --format=csv

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB
compute_cap
7.5


## 1. Install build prerequisites

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq git wget curl build-essential cmake     libblas-dev liblapack-dev libfftw3-dev openmpi-bin libopenmpi-dev     environment-modules

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package environment-modules.
(Reading database ... 126952 files and directories currently installed.)
Preparing t

## 2. Install NVIDIA HPC SDK 25.5

QE's official GPU build requires the NVIDIA HPC SDK/nvfortran for the current NVIDIA/OpenACC GPU path. We use HPC SDK 25.5 because it bundles CUDA 12.9.

In [ ]:
!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK   | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg

!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /'   | sudo tee /etc/apt/sources.list.d/nvhpc.list > /dev/null

!sudo apt-get update -qq
!sudo apt-get install -y nvhpc-25-5

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  nvhpc-25-5
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 5,563 MB of archives.
After this operation, 18.6 GB of additional disk space will be used.
Get:1 https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64  nvhpc-25-5 25.5-0 [5,563 MB]
Fetched 5,563 MB in 24min 15s (3,823 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot b

In [ ]:
%%bash
set -e

export NVCOMPILERS=/opt/nvidia/hpc_sdk
export NVARCH=Linux_x86_64
export NVHPC=$NVCOMPILERS/$NVARCH/25.5

export PATH=$NVHPC/compilers/bin:$NVHPC/comm_libs/mpi/bin:$PATH

echo "NVHPC=$NVHPC"
echo "nvfortran:"
which nvfortran
nvfortran --version | head -4

echo
echo "Available CUDA toolkits:"
ls -d "$NVHPC"/cuda/* 2>/dev/null || true

echo
echo "GPU/compiler information:"
nvaccelinfo | head -35

NVHPC=/opt/nvidia/hpc_sdk/Linux_x86_64/25.5
nvfortran:
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/compilers/bin/nvfortran

nvfortran 25.5-0 64-bit target on x86-64 Linux -tp skylake-avx512 
NVIDIA Compilers and Tools
Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

Available CUDA toolkits:
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/12.9
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/bin
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/include
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/lib64
/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/nvvm

GPU/compiler information:

CUDA Driver Version:           13000
NVRM version:                  NVIDIA UNIX Open Kernel Module for x86_64  580.82.07  Release Build  (builder@28e54e79972f)  Thu Apr 30 1
8:50:30 UTC 2026

Device Number:                 0
Device Name:                   Tesla T4
Device Revision Number:        7.5
Global Memory Size:            15637086208
Number of Multiprocessors:     40
Concurrent Copy and Execution: Yes
Tota

## 3. Set the CUDA environment

This selects CUDA 12.9 from HPC SDK 25.5.

In [ ]:
%%bash
set -e

export NVCOMPILERS=/opt/nvidia/hpc_sdk
export NVARCH=Linux_x86_64
export NVHPC=$NVCOMPILERS/$NVARCH/25.5
export NVHPC_CUDA_HOME=$NVHPC/cuda/12.9

export PATH=$NVHPC/compilers/bin:$NVHPC/comm_libs/mpi/bin:$PATH
export LD_LIBRARY_PATH=$NVHPC/cuda/12.9/lib64:$NVHPC/math_libs/lib64:$NVHPC/compilers/lib:$NVHPC/comm_libs/mpi/lib:${LD_LIBRARY_PATH:-}

echo "NVHPC_CUDA_HOME=$NVHPC_CUDA_HOME"
nvaccelinfo | grep -E 'Default Target|CUDA Driver Version' || true

NVHPC_CUDA_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/12.9
CUDA Driver Version:           13000
Default Target:                cc75


## 4. Download and build Quantum ESPRESSO 7.6 for the T4

T4 = compute capability 7.5, so QE is compiled with `--with-cuda-cc=75`.

For this 30-atom test, Scalapack is disabled; a single GPU is used.

In [ ]:
%%bash
set -e

cd /content

rm -rf q-e-qe-7.6 q-e-qe-7.6.tar.gz

# Download QE 7.6
wget -q -O q-e-qe-7.6.tar.gz \
    https://gitlab.com/QEF/q-e/-/archive/qe-7.6/q-e-qe-7.6.tar.gz

tar -xzf q-e-qe-7.6.tar.gz

# NVIDIA HPC SDK
export NVCOMPILERS=/opt/nvidia/hpc_sdk
export NVARCH=Linux_x86_64
export NVHPC=$NVCOMPILERS/$NVARCH/25.5

export NVHPC_CUDA_HOME=$NVHPC/cuda/12.9
export NVCOMPILER_COMM_LIBS_HOME=$NVHPC/comm_libs/12.9

export PATH=$NVHPC/compilers/bin:$NVHPC/comm_libs/mpi/bin:$PATH

export LD_LIBRARY_PATH=$NVHPC_CUDA_HOME/lib64:\
$NVHPC/comm_libs/12.9/lib:\
$NVHPC/math_libs/12.9/lib:\
$NVHPC/compilers/lib:\
$NVHPC/comm_libs/mpi/lib:\
${LD_LIBRARY_PATH:-}

# NVIDIA compilers
export FC=nvfortran
export F90=nvfortran
export CC=nvc
export CXX=nvc++
export MPIF90=$NVHPC/comm_libs/mpi/bin/mpif90

# Install required numerical libraries
sudo apt-get update -qq

sudo apt-get install -y -qq \
    libfftw3-dev \
    libblas-dev \
    liblapack-dev

# Avoid Colab MKL interference
unset MKLROOT
unset MKL_HOME
unset MKL_ROOT
unset CPATH
unset C_INCLUDE_PATH
unset CPLUS_INCLUDE_PATH
unset FPATH
unset LIBRARY_PATH

cd /content/q-e-qe-7.6

# Configure
./configure \
    --with-gpu=cuda \
    --with-cuda=$NVHPC_CUDA_HOME \
    --with-cuda-cc=75 \
    --with-cuda-runtime=12.9 \
    --enable-openmp \
    --without-scalapack \
    F90=nvfortran \
    MPIF90=$MPIF90 \
    CC=nvc \
    CXX=nvc++

# Force FFTW3 + system BLAS/LAPACK
sed -i 's/-D__DFTI//g' make.inc
sed -i 's|-I/opt/intel/mkl/include||g' make.inc

sed -i 's/^DFLAGS[[:space:]]*=.*/DFLAGS = -D__PGI -D__CUDA -D__FFTW3 -D__MPI -D__MPI_MODULE/' make.inc

sed -i 's|^BLAS_LIBS.*|BLAS_LIBS = -lblas|' make.inc
sed -i 's|^LAPACK_LIBS.*|LAPACK_LIBS = -llapack|' make.inc

# IMPORTANT: include FFTW threaded library
sed -i 's|^FFT_LIBS.*|FFT_LIBS = -lfftw3 -lfftw3_threads -lpthread|' make.inc

echo
echo "===== FINAL CONFIGURATION ====="
grep -E '^DFLAGS|^IFLAGS|^BLAS_LIBS|^LAPACK_LIBS|^FFT_LIBS' make.inc

echo
echo "===== CHECK FFTW THREAD LIBRARY ====="
ldconfig -p | grep fftw3_threads || true

echo
echo "===== BUILDING QE pw.x ====="

make -j2 pw

echo
echo "===== QE BUILD COMPLETE ====="

ls -lh bin/pw.x

echo
echo "===== TESTING pw.x ====="

bin/pw.x -h | head -20

in-source build
checking build system type... x86_64-pc-linux-gnu
checking ARCH... x86_64
checking setting AR... ... ar
checking setting ARFLAGS... ... ruv
checking whether the Fortran compiler works... yes
checking for Fortran compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether we are using the GNU Fortran compiler... no
checking whether nvfortran accepts -g... yes
checking for Fortran flag to compile .f90 files... none
checking for /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/comm_libs/mpi/bin/mpif90... no
checking whether we are using the GNU Fortran compiler... no
checking whether  accepts -g... no
checking version of /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/comm_libs/mpi/bin/mpif90... nvfortran 25.5-0
setting F90... nvfortran
setting MPIF90... /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/comm_libs/mpi/bin/mpif90
checking whether we are using the GNU C compiler... ye

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
configure: WARNING: --with-cuda is obsolete, use --with-gpu=cuda instead
configure: WARNING: MPIF90 not found: using MPIF90 anyway
configure: WARNING: Check whether user-supplied F90 is consistent with MPIF90 !!! 
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'develo

## 5. Check that QE really has GPU support

In [ ]:
%%bash
cd /content/q-e-qe-7.6
grep -i "GPU acceleration" make.inc || true
echo
ldd bin/pw.x | grep -Ei 'cuda|cudart|cublas|cusolver' || true


	libcudaforwraprand.so => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/compilers/lib/libcudaforwraprand.so (0x000078eda8200000)
	libcusolver.so.11 => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/math_libs/12.9/lib64/libcusolver.so.11 (0x000078ed8c200000)
	libnvJitLink.so.12 => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/12.9/lib64/libnvJitLink.so.12 (0x000078ed86579000)
	libcublas.so.12 => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/math_libs/12.9/lib64/libcublas.so.12 (0x000078ed7fe00000)
	libcublasLt.so.12 => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/math_libs/12.9/lib64/libcublasLt.so.12 (0x000078ed4dc00000)
	libcudaforwrapblas.so => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/compilers/lib/libcudaforwrapblas.so (0x000078ed4d800000)
	libcudaforwrapblas117.so => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/compilers/lib/libcudaforwrapblas117.so (0x000078ed4d400000)
	libcudart.so.12 => /opt/nvidia/hpc_sdk/Linux_x86_64/25.5/cuda/12.9/lib64/libcudart.so.12 (0x000078ed4d000000)
	libcudafor_128.so => /opt/nvidia/hpc_sdk/Linu

## 6. Upload the input package prepared earlier

Download `Li3SiNi3O8_QE_inputs.tar.gz` from the ChatGPT message, then upload it in the next cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving Li3SiNi3O8_QE_inputs.tar.gz to Li3SiNi3O8_QE_inputs.tar.gz
Uploaded: ['Li3SiNi3O8_QE_inputs.tar.gz']


In [ ]:
%%bash
set -e
cd /content
rm -rf Li3SiNi3O8_QE
tar -xzf Li3SiNi3O8_QE_inputs.tar.gz
ls -lh Li3SiNi3O8_QE

total 20K
-rwxr-xr-x 1 root 1001  337 Sep 16 10:11 download_pseudos.sh
-rw-r--r-- 1 root 1001 3.1K Sep 16 10:11 final_scf.in
-rw-r--r-- 1 root 1001 2.1K Sep 16 10:11 POSCAR
-rw-r--r-- 1 root 1001 1.2K Sep 16 10:11 README.txt
-rw-r--r-- 1 root 1001 3.1K Sep 16 10:11 scf.in


## 7. Download the PBE PAW pseudopotentials

These are the exact four PSlibrary files used in the prepared QE input:
- Li.pbe-s-kjpaw_psl.1.0.0.UPF
- Si.pbe-n-kjpaw_psl.1.0.0.UPF
- Ni.pbe-spn-kjpaw_psl.1.0.0.UPF
- O.pbe-n-kjpaw_psl.1.0.0.UPF

In [ ]:
%%bash
set -e

cd /content/Li3SiNi3O8_QE
mkdir -p pseudo

wget -q -O pseudo/Li.pbe-s-kjpaw_psl.1.0.0.UPF   https://pseudopotentials.quantum-espresso.org/upf_files/Li.pbe-s-kjpaw_psl.1.0.0.UPF

wget -q -O pseudo/Si.pbe-n-kjpaw_psl.1.0.0.UPF   https://pseudopotentials.quantum-espresso.org/upf_files/Si.pbe-n-kjpaw_psl.1.0.0.UPF

wget -q -O pseudo/Ni.pbe-spn-kjpaw_psl.1.0.0.UPF   https://pseudopotentials.quantum-espresso.org/upf_files/Ni.pbe-spn-kjpaw_psl.1.0.0.UPF

wget -q -O pseudo/O.pbe-n-kjpaw_psl.1.0.0.UPF   https://pseudopotentials.quantum-espresso.org/upf_files/O.pbe-n-kjpaw_psl.1.0.0.UPF

ls -lh pseudo

total 5.0M
-rw-r--r-- 1 root 1001 770K Nov 16  2018 Li.pbe-s-kjpaw_psl.1.0.0.UPF
-rw-r--r-- 1 root 1001 1.8M Nov 16  2018 Ni.pbe-spn-kjpaw_psl.1.0.0.UPF
-rw-r--r-- 1 root 1001 855K Nov 16  2018 O.pbe-n-kjpaw_psl.1.0.0.UPF
-rw-r--r-- 1 root 1001 1.7M Nov 16  2018 Si.pbe-n-kjpaw_psl.1.0.0.UPF


## 8. Run the magnetic SCF calculation

Important:
- This is spin-polarized (`nspin=2`).
- Ni is split into `Ni_up` and `Ni_dn` so the six Ni sites start with a 3-down/3-up pattern.
- Ni Hubbard U = 6.2 eV.
- The input uses 120 Ry wavefunction cutoff and 480 Ry charge-density cutoff.
- The calculation uses the 6x4x4 k-point mesh from the prepared input.

For one T4 GPU, start with one MPI rank. QE's GPU build should print `GPU acceleration is ACTIVE.` near the beginning of the output.

Before running below cell make changes in "scf.in",

1,

* ecutwfc = 120 Ry
* ecutrho = 480 Ry

to

* ecutwfc = 80
* ecutrho = 320

2,

* K_POINTS automatic
* 6 4 4 0 0 0

to

* K_POINTS automatic
* 2 2 2 0 0 0

3,

* Ni_up
* Ni_dn

to

* Niu
* Nid

4,

* conv_thr = 1.0d-8
* electron_maxstep = 200
* mixing_beta = 0.30

to

* conv_thr = 1.0d-6
* mixing_beta = 0.2
* electron_maxstep = 100


Available RAM

In [ ]:
%%bash
echo "===== RAM ====="
free -h

echo
echo "===== GPU ====="
nvidia-smi --query-gpu=name,memory.used,memory.free,memory.total \
--format=csv

===== RAM =====
               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.2Gi       1.9Gi       193Mi       8.2Gi       9.5Gi
Swap:             0B          0B          0B

===== GPU =====
name, memory.used [MiB], memory.free [MiB], memory.total [MiB]
Tesla T4, 1807 MiB, 13106 MiB, 15360 MiB


MPI run

In [ ]:
%%bash

export NVCOMPILERS=/opt/nvidia/hpc_sdk
export NVARCH=Linux_x86_64
export NVHPC=$NVCOMPILERS/$NVARCH/25.5

export NVHPC_CUDA_HOME=$NVHPC/cuda/12.9
export NVCOMPILER_COMM_LIBS_HOME=$NVHPC/comm_libs/12.9

export PATH=$NVHPC/compilers/bin:$NVHPC/comm_libs/mpi/bin:/content/q-e-qe-7.6/bin:$PATH

export LD_LIBRARY_PATH=$NVHPC_CUDA_HOME/lib64:\
$NVHPC/comm_libs/12.9/lib:\
$NVHPC/math_libs/12.9/lib:\
$NVHPC/compilers/lib:\
$NVHPC/comm_libs/mpi/lib:\
${LD_LIBRARY_PATH:-}

export OMP_NUM_THREADS=1

cd /content/Li3SiNi3O8_QE

rm -rf tmp
mkdir -p tmp

mpirun --allow-run-as-root -np 1 \
    /content/q-e-qe-7.6/bin/pw.x \
    -in scf.in 2>&1 | tee scf.out

Process is terminated.


## 9. Inspect SCF convergence and magnetization

In [ ]:
%%bash
cd /content/Li3SiNi3O8_QE

echo "=== GPU status ==="
grep -i "GPU acceleration" scf.out || true

echo
echo "=== SCF convergence ==="
grep -i "convergence has been achieved" scf.out || true

echo
echo "=== Final total energy ==="
grep -E "!    total energy" scf.out | tail -5 || true

echo
echo "=== Total magnetization ==="
grep -i "total magnetization" scf.out | tail -5 || true

echo
echo "=== Absolute magnetization ==="
grep -i "absolute magnetization" scf.out | tail -5 || true

echo
echo "=== Final iterations ==="
grep -E "iteration #|estimated scf accuracy" scf.out | tail -15 || true

=== GPU status ===
     GPU acceleration is ACTIVE.  1 visible GPUs per MPI rank

=== SCF convergence ===

=== Final total energy ===

=== Total magnetization ===
     total magnetization       =    -0.03 Bohr mag/cell
     total magnetization       =    -0.05 Bohr mag/cell
     total magnetization       =    -0.02 Bohr mag/cell
     total magnetization       =    -0.03 Bohr mag/cell
     total magnetization       =    -0.04 Bohr mag/cell

=== Absolute magnetization ===
     absolute magnetization    =    21.28 Bohr mag/cell
     absolute magnetization    =    21.65 Bohr mag/cell
     absolute magnetization    =    22.71 Bohr mag/cell
     absolute magnetization    =    22.18 Bohr mag/cell
     absolute magnetization    =    21.65 Bohr mag/cell

=== Final iterations ===
     estimated scf accuracy    <       5.13117915 Ry
     iteration # 94     ecut=    80.00 Ry     beta= 0.20
     estimated scf accuracy    <       3.63327262 Ry
     iteration # 95     ecut=    80.00 Ry     beta= 0.20

## 10. Print atomic magnetic moments (QE output)

QE prints atomic spin/magnetization information in the SCF output when requested by the version/build.

In [ ]:
%%bash
cd /content/Li3SiNi3O8_QE

echo "=== Magnetization-related output ==="
grep -in -E "magnetization|site magnetic|atomic magnetic" scf.out | tail -80 || true

=== Magnetization-related output ===
21102:     Atomic magnetic moment for atom   9 =  -0.85825
21135:     Atomic magnetic moment for atom  10 =  -0.99593
21168:     Atomic magnetic moment for atom  11 =  -1.09411
21201:     Atomic magnetic moment for atom  12 =   1.04042
21234:     Atomic magnetic moment for atom  13 =   1.05990
21267:     Atomic magnetic moment for atom  14 =   1.14731
21310:     total magnetization       =    -0.01 Bohr mag/cell
21311:     absolute magnetization    =    21.70 Bohr mag/cell
21331:     Atomic magnetic moment for atom   9 =  -0.88032
21364:     Atomic magnetic moment for atom  10 =  -1.00256
21397:     Atomic magnetic moment for atom  11 =  -1.08882
21430:     Atomic magnetic moment for atom  12 =   1.05806
21463:     Atomic magnetic moment for atom  13 =   1.06416
21496:     Atomic magnetic moment for atom  14 =   1.15689
21539:     total magnetization       =    -0.03 Bohr mag/cell
21540:     absolute magnetization    =    21.77 Bohr mag/cell
21560: 

## 11. Run the tighter final SCF only after the first SCF converges

The supplied `final_scf.in` restarts from the converged directory and uses a tighter threshold.

Before running make changes in final_scf.in:

1,

* ecutwfc = 120.0
* ecutrho = 480.0

to

* ecutwfc = 80
* ecutrho = 320

2,

* conv_thr = 1.0d-10
* electron_maxstep = 200
* mixing_beta = 0.30

to

* conv_thr = 1.0d-6
* electron_maxstep = 150
* mixing_beta = 0.20

3,

* Ni_up
* Ni_dn

to

* Niu
* Nid

4,

* K_POINTS automatic
* 6 4 4 0 0 0

to

* K_POINTS automatic
* 2 2 2 0 0 0

5,

Add in &SYSYTEM

* starting_magnetization(3) = -0.7
* starting_magnetization(4) =  0.7

In [ ]:
%%bash
set -o pipefail

export NVHPCSDK_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/25.5

source $NVHPCSDK_HOME/comm_libs/12.9/hpcx/hpcx-2.22.1/hpcx-init.sh
hpcx_load

export PATH=$NVHPCSDK_HOME/compilers/bin:/content/q-e-qe-7.6/bin:$PATH
export LD_LIBRARY_PATH=$NVHPCSDK_HOME/cuda/12.9/lib64:$NVHPCSDK_HOME/compilers/lib:${LD_LIBRARY_PATH:-}

export OMP_NUM_THREADS=1

cd /content/Li3SiNi3O8_QE

rm -f final_scf.out

echo "===== STARTING FINAL SCF ====="

mpirun --allow-run-as-root -np 1 \
    /content/q-e-qe-7.6/bin/pw.x \
    -in final_scf.in 2>&1 | tee final_scf.out

===== STARTING FINAL SCF =====

     Program PWSCF v.7.6 starts on 19Sep2026 at  9: 3: 2 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Parallel version (MPI & OpenMP), running on       1 processor cores
     Number of MPI processes:                 1
     Threads/MPI process:                     1

     MPI processes distributed on     1 nodes
     2055 MiB available memory on the printing compute node when the environment starts
 
     Reading input from final_scf.in

     Current dimensions of program PWSCF are:
     Max numb

## 12. Download the results

The most important files are:
- `scf.out`
- `final_scf.out`
- `tmp/` (restart data)

Download the two text outputs first.

In [ ]:
from google.colab import files
files.download('/content/Li3SiNi3O8_QE/scf.out')
files.download('/content/Li3SiNi3O8_QE/final_scf.out')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Expected scientific workflow after this

Do NOT jump directly to a band calculation.

First confirm:
1. SCF converged.
2. GPU acceleration was active.
3. Total magnetization is approximately compensated.
4. The six Ni sites retain the intended 3-up/3-down magnetic solution.
5. The final energy is stable.

Then we can prepare the QE NSCF/band/DOS calculations needed to examine the altermagnetic electronic structure.

## 13. SCF convergence

Changes in final_scf.in file

* electron_maxstep = 300
* mixing_beta = 0.10
* conv_thr = 1.0d-6

In [ ]:
cd /content/Li3SiNi3O8_QE

/content/Li3SiNi3O8_QE


In [ ]:
!export HPCX_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/25.5/comm_libs/12.9/hpcx/hpcx-2.22.1 && \
source $HPCX_HOME/hpcx-init.sh && \
hpcx_load && \
export OMPI_ALLOW_RUN_AS_ROOT=1 && \
export OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1 && \
mpirun -np 1 /content/q-e-qe-7.6/PW/src/pw.x -in final_scf.in 2>&1 | tee final_scf_2.out

Streaming output truncated to the last 5000 lines.
     eigenvectors (columns):
      -0.035 -0.097 -0.833  0.527  0.135
       0.385  0.795  0.163  0.365  0.246
      -0.695  0.424  0.003  0.179 -0.552
       0.536 -0.218  0.108  0.352 -0.728
      -0.283 -0.362  0.518  0.658  0.295
     occupation matrix ns (before diag.):
       0.994  0.004 -0.006  0.004 -0.002
       0.004  0.953  0.066 -0.052  0.030
      -0.006  0.066  0.871  0.097 -0.050
       0.004 -0.052  0.097  0.923  0.038
      -0.002  0.030 -0.050  0.038  0.974
     SPIN  2
     eigenvalues:
       0.201  0.324  0.994  0.995  0.997
     eigenvectors (columns):
      -0.003  0.027 -0.455 -0.852  0.259
       0.753  0.311  0.471 -0.153  0.304
       0.303 -0.710  0.220 -0.299 -0.516
      -0.248  0.585  0.306 -0.334 -0.626
      -0.530 -0.238  0.655 -0.226  0.428
     occupation matrix ns (before diag.):
       0.994 -0.003  0.014 -0.012  0.003
      -0.003  0.480 -0.033  0.026  0.366
       0.014 -0.033  0.584  0.339  0.0

Checking result

In [ ]:
!tail -35 final_scf_2.out

     h_psi:calbec :      0.39s CPU      0.41s WALL (    5244 calls)
     vloc_psi     :    558.97s CPU    562.48s WALL (    5244 calls)
     add_vuspsi   :    266.79s CPU    268.75s WALL (    5244 calls)
     vhpsi        :    181.24s CPU    182.53s WALL (    5244 calls)

     General routines
     calbec       :     84.76s CPU     56.13s WALL (   61032 calls)
     fft          :      1.21s CPU      1.23s WALL (    2627 calls)
     ffts         :      0.12s CPU      0.12s WALL (     246 calls)
     fftw         :    581.50s CPU    585.66s WALL (   94928 calls)
     davcio       :      0.00s CPU      0.21s WALL (      16 calls)
 
     Parallel routines

     Hubbard U routines
     new_ns       :    105.89s CPU     75.63s WALL (     123 calls)
     vhpsi        :    181.24s CPU    182.53s WALL (    5244 calls)
     force_hub    :    127.73s CPU    128.40s WALL (       1 calls)
     stres_hub    :    152.35s CPU    153.05s WALL (       1 calls)

     PAW routines
     PAW_pot      :    1

## 14. Downloading the necessary files in compressed format

In [ ]:
# %%bash

# mkdir -p Li3SiNi3O8_band_backup

# # Main input and outputs
# cp final_scf.in Li3SiNi3O8_band_backup/ 2>/dev/null || true
# cp final_scf_2.out Li3SiNi3O8_band_backup/ 2>/dev/null || true
# cp final_scf.out Li3SiNi3O8_band_backup/ 2>/dev/null || true
# cp scf.in Li3SiNi3O8_band_backup/ 2>/dev/null || true
# cp scf_result.out Li3SiNi3O8_band_backup/ 2>/dev/null || true
# cp POSCAR Li3SiNi3O8_band_backup/ 2>/dev/null || true

# # Preserve the complete QE restart data
# cp -r tmp/Li3SiNi3O8.save Li3SiNi3O8_band_backup/

# # Preserve the exact pseudopotentials
# cp -r pseudo Li3SiNi3O8_band_backup/

# # Create a small record of the backup
# cat > Li3SiNi3O8_band_backup/README.txt << 'EOF'
# Li3SiNi3O8 QE band-structure restart backup

# QE version used:
# 7.6

# Important restart directory:
# tmp/Li3SiNi3O8.save/

# Pseudopotentials:
# Li.pbe-s-kjpaw_psl.1.0.0.UPF
# Si.pbe-n-kjpaw_psl.1.0.0.UPF
# Ni.pbe-spn-kjpaw_psl.1.0.0.UPF
# O.pbe-n-kjpaw_psl.1.0.0.UPF

# Final SCF settings used:
# ecutwfc = 80 Ry
# ecutrho = 320 Ry
# 2 x 2 x 2 k-mesh
# nspin = 2
# U(Ni) = 6.2 eV
# starting magnetization:
# Niu = -0.7
# Nid = +0.7

# Next calculation:
# non-SCF band calculation followed by bands.x
# EOF

# # Show size first
# du -sh Li3SiNi3O8_band_backup

# # Compress everything
# tar -czf Li3SiNi3O8_band_backup.tar.gz Li3SiNi3O8_band_backup

# ls -lh Li3SiNi3O8_band_backup.tar.gz

Downloading the compressed file

In [ ]:
# from google.colab import files
# files.download("Li3SiNi3O8_band_backup.tar.gz")